# 01 - Teacher & Student baseline

In [1]:
import sys; sys.path.append("..")
%load_ext autoreload
%autoreload 2

import torch
import matplotlib.pyplot as plt
from src.data import get_dataloaders
from src.models import build_model
from src.metrics import count_parameters, model_size_mb, save_result
from src.train import get_device, init_weights, fit, evaluate, overfit_one_batch

device = get_device()
torch.backends.cudnn.benchmark = True

train_loader, val_loader, test_loader = get_dataloaders("../data", batch_size=128)

Device: cpu


In [2]:
# Sanity check: overfit su un solo batch (train) per verificare che la loss scenda  (e il training quindi funzioni)
for name in ["teacher", "student"]:
    torch.manual_seed(0)
    m = build_model(name)
    init_weights(m)
    print(f"--- {name} ---")
    overfit_one_batch(m, train_loader, device)

--- teacher ---


c:\Users\Pietr\Desktop\Model-Compression\.venv\lib\site-packages\torch\utils\data\dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


step   0 | loss 2.5312
step  20 | loss 0.0962
step  40 | loss 0.0061
step  60 | loss 0.0045
step  80 | loss 0.0004
step  99 | loss 0.0004
--- student ---
step   0 | loss 2.3111
step  20 | loss 1.6453
step  40 | loss 0.1680
step  60 | loss 0.0341
step  80 | loss 0.0128
step  99 | loss 0.0077


In [3]:
# Baseline training: traina il modello per x epoche, salva il miglior checkpoint sul validation set e registra i risultati in checkpoints e results.json
# funzioni helper per evitare replicazione codice
def run_baseline(name, epochs=30):
    torch.manual_seed(0)
    model = build_model(name)
    init_weights(model)
    model.to(device)

    history, train_time = fit(model, train_loader, val_loader, device, epochs=epochs,
                              ckpt_path=f"../checkpoints/{name}_baseline.pt")

    _, val_acc = evaluate(model, val_loader, device)
    _, test_acc = evaluate(model, test_loader, device)
    c = count_parameters(model)
    print(f"\n{name}: val {val_acc * 100:.2f}% | test {test_acc * 100:.2f}% | "
          f"{c['total']:,} params | {model_size_mb(model):.2f} MB | {train_time / 60:.1f} min")

    save_result(f"{name}_baseline", val_acc=val_acc, test_acc=test_acc,
                params=c["total"], size_mb=model_size_mb(model),
                train_time_s=train_time, epochs=epochs)
    return model, history

In [11]:
# Addestramento del teacher
teacher, h_teacher = run_baseline("teacher",epochs=30)

ep   1/30 | train loss 2.238 acc 16.01 | val loss 1.982 acc 20.74 (best 20.74) | lr 0.0500
ep   2/30 | train loss 1.873 acc 26.58 | val loss 1.643 acc 35.14 (best 35.14) | lr 0.0499
ep   3/30 | train loss 1.564 acc 42.75 | val loss 1.616 acc 47.98 (best 47.98) | lr 0.0495
ep   4/30 | train loss 1.250 acc 56.49 | val loss 1.364 acc 54.40 (best 54.40) | lr 0.0488
ep   5/30 | train loss 1.032 acc 64.78 | val loss 1.030 acc 65.18 (best 65.18) | lr 0.0478
ep   6/30 | train loss 0.893 acc 69.82 | val loss 0.845 acc 71.14 (best 71.14) | lr 0.0467
ep   7/30 | train loss 0.799 acc 72.95 | val loss 0.896 acc 70.62 (best 71.14) | lr 0.0452
ep   8/30 | train loss 0.729 acc 75.76 | val loss 0.734 acc 74.32 (best 74.32) | lr 0.0436
ep   9/30 | train loss 0.665 acc 77.88 | val loss 0.644 acc 77.66 (best 77.66) | lr 0.0417
ep  10/30 | train loss 0.609 acc 79.77 | val loss 0.605 acc 78.92 (best 78.92) | lr 0.0397
ep  11/30 | train loss 0.569 acc 81.10 | val loss 0.608 acc 79.08 (best 79.08) | lr 0.0375

In [4]:
# Addestramento dello student
student, h_student = run_baseline("student",epochs=30)

ep   1/30 | train loss 1.711 acc 35.98 | val loss 1.406 acc 48.20 (best 48.20) | lr 0.0500
ep   2/30 | train loss 1.372 acc 51.12 | val loss 1.456 acc 47.64 (best 48.20) | lr 0.0499
ep   3/30 | train loss 1.208 acc 57.49 | val loss 1.172 acc 58.26 (best 58.26) | lr 0.0495
ep   4/30 | train loss 1.103 acc 61.64 | val loss 1.050 acc 62.68 (best 62.68) | lr 0.0488
ep   5/30 | train loss 1.034 acc 64.16 | val loss 1.038 acc 63.40 (best 63.40) | lr 0.0478
ep   6/30 | train loss 0.980 acc 66.38 | val loss 0.965 acc 65.40 (best 65.40) | lr 0.0467
ep   7/30 | train loss 0.940 acc 67.89 | val loss 0.894 acc 68.94 (best 68.94) | lr 0.0452
ep   8/30 | train loss 0.911 acc 68.98 | val loss 1.029 acc 64.74 (best 68.94) | lr 0.0436
ep   9/30 | train loss 0.872 acc 70.25 | val loss 0.903 acc 68.46 (best 68.94) | lr 0.0417
ep  10/30 | train loss 0.847 acc 71.09 | val loss 0.891 acc 68.98 (best 68.98) | lr 0.0397
ep  11/30 | train loss 0.827 acc 72.22 | val loss 0.806 acc 72.06 (best 72.06) | lr 0.0375

In [5]:
from src.metrics import profile_model, update_result

# Carica i modelli addestrati e profila le loro prestazioni (MACs, parametri, dimensione, latenza) [salva in results.json]
for name in ["teacher", "student"]:
    model = build_model(name)
    model.load_state_dict(torch.load(f"../checkpoints/{name}_baseline.pt", map_location="cpu"))
    prof = profile_model(model, dtype="fp32")
    update_result(f"{name}_baseline", **prof)
    print(name, {k: round(v, 3) if isinstance(v, float) else v for k, v in prof.items()})

teacher {'params': 15530762, 'size_mb': 62.165, 'macs': 215689216, 'energy_uj': 992.17, 'peak_act_kb': 327.68, 'latency_ms_median': 7.997, 'latency_ms_std': 0.226}
student {'params': 245226, 'size_mb': 1.004, 'macs': 3564800, 'energy_uj': 16.398, 'peak_act_kb': 45.056, 'latency_ms_median': 0.69, 'latency_ms_std': 0.087}
